In [1]:
import requests
import pandas as pd

In [2]:
api_base_url = "https://anapioficeandfire.com/api"

In [3]:
#  Все книги
books = []
page = 1
while True:
    resp = requests.get(
        f'{api_base_url}/books',
        params={'page': page, 'pageSize': 50}
    )
    data = resp.json()
    if not data:
        break
    books.extend(data)
    if 'rel="next"' not in resp.headers.get('Link', ''):
        break
    page += 1

print(f'Книг: {len(books)}')

# дома
houses = []
page = 1
while True:
    resp = requests.get(
        f'{api_base_url}/houses',
        params={'page': page, 'pageSize': 50}
    )
    data = resp.json()
    if not data:
        break
    houses.extend(data)
    if 'rel="next"' not in resp.headers.get('Link', ''):
        break
    page += 1

print(f'Домов: {len(houses)}')

# Дома с девизом
houses_with_titles = []
page = 1
while True:
    resp = requests.get(
        f'{api_base_url}/houses',
        params={'page': page, 'pageSize': 50, 'hasTitles': 'true'}
    )
    data = resp.json()
    if not data:
        break
    houses_with_titles.extend(data)
    if 'rel="next"' not in resp.headers.get('Link', ''):
        break
    page += 1

print(f'Домов с титулами/девизом: {len(houses_with_titles)}')

# Сохраняем в DataFrame
import pandas as pd

df_books = pd.DataFrame(books)
df_houses = pd.DataFrame(houses)
df_houses_with_titles = pd.DataFrame(houses_with_titles)


Книг: 12
Домов: 444
Домов с титулами/девизом: 165


In [4]:
pip install psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 43.9 MB/s eta 0:00:00


In [6]:
import psycopg2

In [9]:
# Подключение к БД
conn = psycopg2.connect(
    host="hh-pgsql-public.ebi.ac.uk",
    database="pfmegrnargs",
    user="reader",
    password="NWDMCE5xdipIjRrp",
    port=5432,
)
print("Подключение установлено")

Подключение установлено


In [10]:
# Получение 10 строк из таблицы rnc_database
cursor = conn.cursor()
cursor.execute("SELECT * FROM rnc_database LIMIT 10")
rows = cursor.fetchall()

UndefinedTable: relation "rnc_database" does not exist
LINE 1: SELECT * FROM rnc_database LIMIT 10
                      ^


таблица rnc_database не сущетсвует

In [22]:
import psycopg2
import pandas as pd

# Подключение
conn = psycopg2.connect(
    host="hh-pgsql-public.ebi.ac.uk",
    dbname="pfmegrnargs",
    user="reader",
    password="NWDMCE5xdipIjRrp",
)
cursor = conn.cursor()

# ── Диагностика: ищем таблицу rnc_database во всех схемах ──
cursor.execute("""
    SELECT schemaname, tablename
    FROM pg_catalog.pg_tables
    WHERE tablename LIKE '%rnc_database%'
""")
print("Найденные таблицы:")
for row in cursor.fetchall():
    print(row)

# ── Основной запрос с полным именем схемы ──────────────────
cursor.execute("SELECT * FROM rnacen.rnc_database LIMIT 10")
rows = cursor.fetchall()
column_names = [desc[0] for desc in cursor.description]
df = pd.DataFrame(rows, columns=column_names)
print("\nПервые 10 строк из rnc_database:")
print(df)

# ── Выборка нужных столбцов ────────────────────────────────
cursor.execute(
    "SELECT display_name, num_sequences, num_organisms, url "
    "FROM rnacen.rnc_database LIMIT 10"
)
rows_selected = cursor.fetchall()
df_selected = pd.DataFrame(
    rows_selected,
    columns=["display_name", "num_sequences", "num_organisms", "url"],
)
print("\nВыбранные столбцы:")
print(df_selected)

cursor.close()
conn.close()


Найденные таблицы:
('rnacen', 'rnc_database')
('rnacen', 'rnc_database_json_stats')
('rnacen', 'rnc_database_references')
('rnacen_django_test', 'rnc_database_json_stats')
('rnacen_django_test', 'rnc_database')


InsufficientPrivilege: permission denied for schema rnacen
LINE 1: SELECT * FROM rnacen.rnc_database LIMIT 10
                      ^


In [23]:
import psycopg2

conn = psycopg2.connect(
    host="hh-pgsql-public.ebi.ac.uk",
    dbname="pfmegrnargs",
    user="reader",
    password="NWDMCE5xdipIjRrp",
)
cursor = conn.cursor()

# 1. Текущий search_path
cursor.execute("SHOW search_path")
print("search_path:", cursor.fetchone())

# 2. Схемы, к которым есть доступ
cursor.execute("""
    SELECT nspname
    FROM pg_namespace
    WHERE has_schema_privilege(current_user, nspname, 'USAGE')
    ORDER BY nspname
""")
print("\nДоступные схемы:")
for row in cursor.fetchall():
    print(" ", row[0])

# 3. Views с именем rnc_database (во всех схемах)
cursor.execute("""
    SELECT schemaname, viewname
    FROM pg_views
    WHERE viewname LIKE '%rnc_database%'
""")
print("\nViews с rnc_database:")
for row in cursor.fetchall():
    print(" ", row)

# 4. Все объекты (tables + views) в схеме public
cursor.execute("""
    SELECT table_name, table_type
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name
""")
print("\nОбъекты в public:")
for row in cursor.fetchall():
    print(" ", row)

# 5. Попробуем через rnc_database_json_stats — вдруг доступ есть
conn.rollback()
cursor.execute("SELECT * FROM rnacen.rnc_database_json_stats LIMIT 1")
print("\nrnc_database_json_stats (тест):", cursor.fetchone())

cursor.close()
conn.close()


search_path: ('rnacen',)

Доступные схемы:
  information_schema
  pg_catalog

Views с rnc_database:

Объекты в public:
  ('pg_stat_statements_info', 'VIEW')


InsufficientPrivilege: permission denied for schema rnacen
LINE 1: SELECT * FROM rnacen.rnc_database_json_stats LIMIT 1
                      ^


не удается подключиться к таблице rnc_database